# 📊 Análisis Exploratorio de Datos (EDA)
## Proyecto: Modelo de Riesgo Crediticio

Este notebook contiene el análisis exploratorio completo de los datos.

## 1. Importación de Librerías

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Carga de Datos

In [3]:
# Función para cargar datos
def cargar_datos():
    ruta_actual = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
    # Si estamos en el notebook, subimos un nivel desde el directorio actual
    if os.path.basename(ruta_actual) == 'notebooks':
        ruta_proyecto = os.path.dirname(ruta_actual)
    else:
        ruta_proyecto = ruta_actual
    ruta_excel = os.path.join(ruta_proyecto, "Base_de_datos.xlsx")
    return pd.read_excel(ruta_excel)

df = cargar_datos()
print(f"✅ Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Leo-PC\\DATA-SCIENCE\\DSFT09\\M5\\PI_M5\\src\\Base_de_datos.xlsx'

In [ ]:
# Vista inicial
df.head()

## 3. Análisis Univariable

In [ ]:
# Información general
df.info()

In [ ]:
# Estadísticas descriptivas - Numéricas
df.describe()

In [ ]:
# Estadísticas descriptivas - Categóricas
df.describe(include=['object', 'str'])

In [ ]:
# Valores nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print("🔍 Columnas con valores nulos:")
print(nulos)

In [ ]:
# Visualización de nulos
plt.figure(figsize=(12, 6))
nulos.plot(kind='bar')
plt.title('Valores Nulos por Columna')
plt.xlabel('Columnas')
plt.ylabel('Cantidad de Nulos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de la variable objetivo
print("🎯 Distribución de 'Pago_atiempo':")
print(df['Pago_atiempo'].value_counts(normalize=True) * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Pago_atiempo'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Conteo')
axes[0].set_xlabel('Pago a tiempo (1=Sí, 0=No)')

df['Pago_atiempo'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Proporción')
plt.tight_layout()
plt.show()

## 4. Análisis Bivariable

In [ ]:
# Correlación con la variable objetivo
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_with_target = df[numeric_cols].corr()['Pago_atiempo'].sort_values(ascending=False)
print("📊 Correlación con 'Pago_atiempo':")
print(corr_with_target)

In [ ]:
# Top correlaciones positivas
plt.figure(figsize=(10, 6))
top_corr = corr_with_target[1:11]  # Excluir la propia variable
top_corr.plot(kind='barh')
plt.title('Top 10 Variables con Mayor Correlación con Pago_atiempo')
plt.xlabel('Correlación')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de variables numéricas por clase
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
cols_to_plot = ['puntaje', 'edad_cliente', 'salario_cliente', 
                'capital_prestado', 'plazo_meses', 'huella_consulta']

for i, col in enumerate(cols_to_plot):
    row, col_idx = i // 3, i % 3
    for label in [0, 1]:
        data = df[df['Pago_atiempo'] == label][col]
        axes[row, col_idx].hist(data, alpha=0.5, label=f'Pago={label}', bins=30)
    axes[row, col_idx].set_title(col)
    axes[row, col_idx].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Variables categóricas vs target
categorical_cols = ['tipo_laboral', 'tendencia_ingresos']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(categorical_cols):
    # Crear tabla de contingencia
    cross_tab = pd.crosstab(df[col], df['Pago_atiempo'], normalize='index') * 100
    cross_tab.plot(kind='bar', stacked=True, ax=axes[i])
    axes[i].set_title(f'{col} vs Pago_atiempo')
    axes[i].set_ylabel('Porcentaje')
    axes[i].legend(title='Pago a tiempo', labels=['No (0)', 'Sí (1)'])
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Análisis Multivariable

In [ ]:
# Matriz de correlación
plt.figure(figsize=(14, 12))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0, 
            annot=False, fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot de variables más relevantes
top_features = ['puntaje', 'puntaje_datacredito', 'edad_cliente', 
                'salario_cliente', 'Pago_atiempo']
sns.pairplot(df[top_features], hue='Pago_atiempo', diag_kind='kde')
plt.suptitle('Pairplot de Variables Relevantes', y=1.02)
plt.tight_layout()
plt.show()

## 6. Análisis de Data Drift (Simulado)

In [ ]:
# Para simular drift, separamos por fecha
df_sorted = df.sort_values('fecha_prestamo')
split_idx = int(len(df_sorted) * 0.7)  # 70% datos históricos, 30% nuevos

df_historico = df_sorted.iloc[:split_idx]
df_nuevo = df_sorted.iloc[split_idx:]

print(f"📊 Datos históricos: {len(df_historico)} filas")
print(f"📊 Datos nuevos: {len(df_nuevo)} filas")

In [ ]:
# Comparación de distribuciones numéricas
def analyze_drift_numeric(df_old, df_new, column):
    """Analiza drift en una variable numérica"""
    old_mean = df_old[column].mean()
    new_mean = df_new[column].mean()
    old_std = df_old[column].std()
    new_std = df_new[column].std()
    
    mean_change = abs((new_mean - old_mean) / old_mean * 100) if old_mean != 0 else 0
    std_change = abs((new_std - old_std) / old_std * 100) if old_std != 0 else 0
    
    return {
        'columna': column,
        'media_historica': old_mean,
        'media_nueva': new_mean,
        'cambio_media_pct': mean_change,
        'std_historica': old_std,
        'std_nueva': new_std,
        'cambio_std_pct': std_change
    }

In [ ]:
# Analizar drift en variables numéricas clave
key_numeric = ['puntaje', 'edad_cliente', 'salario_cliente', 
               'capital_prestado', 'plazo_meses']

drift_results = []
for col in key_numeric:
    result = analyze_drift_numeric(df_historico, df_nuevo, col)
    drift_results.append(result)

df_drift = pd.DataFrame(drift_results)
df_drift

In [ ]:
# Visualizar drift
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(key_numeric):
    row, col_idx = i // 3, i % 3
    sns.kdeplot(df_historico[col], label='Histórico', ax=axes[row, col_idx])
    sns.kdeplot(df_nuevo[col], label='Nuevo', ax=axes[row, col_idx])
    axes[row, col_idx].set_title(f'{col}')
    axes[row, col_idx].legend()

# Ocultar ejes vacíos
for i in range(len(key_numeric), 6):
    row, col_idx = i // 3, i % 3
    axes[row, col_idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Prueba de Chi-cuadrado para variables categóricas
from scipy.stats import chi2_contingency

def chi2_drift_test(df_old, df_new, col):
    """Prueba chi-cuadrado para drift en variable categórica"""
    # Obtener categorías
    categories = pd.concat([df_old[col], df_new[col]]).unique()
    
    # Crear tabla de contingencia
    old_counts = df_old[col].value_counts().reindex(categories, fill_value=0)
    new_counts = df_new[col].value_counts().reindex(categories, fill_value=0)
    
    contingency = pd.DataFrame([old_counts, new_counts])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    return {
        'columna': col,
        'chi2': chi2,
        'p_value': p_value,
        'drift_detectado': p_value < 0.05
    }


In [ ]:
# Analizar drift en variables categóricas
categorical_cols = ['tipo_laboral', 'tendencia_ingresos']
for col in categorical_cols:
    if col in df.columns:
        result = chi2_drift_test(df_historico, df_nuevo, col)
        print(f"📊 {col}:")
        print(f"   Chi2 = {result['chi2']:.4f}, p-value = {result['p_value']:.4f}")
        print(f"   {'⚠️ Drift detectado' if result['drift_detectado'] else '✅ Sin drift significativo'}\n")

## 7. Resumen de Hallazgos

In [ ]:
print("="*60)
print("📋 RESUMEN DE HALLAZGOS DEL EDA")
print("="*60)

print("\n1️⃣ DISTRIBUCIÓN DEL TARGET:")
print(f"   - {df['Pago_atiempo'].sum()} clientes pagan a tiempo ({df['Pago_atiempo'].mean()*100:.1f}%)")
print(f"   - Desbalanceo significativo: ~5% de casos de impago")

print("\n2️⃣ VARIABLES MÁS RELEVANTES (Correlación):")
top_features = corr_with_target[1:6]
for idx, (col, corr) in enumerate(top_features.items(), 1):
    print(f"   {idx}. {col}: {corr:.4f}")

print("\n3️⃣ VALORES NULOS:")
print(f"   - {len(nulos)} columnas con valores nulos")
print(f"   - Mayor afectación: promedio_ingresos_datacredito ({nulos.max()} nulos)")

print("\n4️⃣ DATA DRIFT DETECTADO:")
print("   - Se detectó drift en algunas variables al comparar datos históricos vs nuevos")
print("   - Recomendación: Monitorear continuamente el desempeño del modelo")

print("\n5️⃣ RECOMENDACIONES PARA EL MODELADO:")
print("   - Usar class_weight='balanced' para manejar el desbalanceo")
print("   - Imputar valores nulos con mediana/moda")
print("   - Considerar features como 'ratio_deuda_ingreso'")
print("   - Monitorear drift de variables como 'puntaje' y 'capital_prestado'")

print("\n✅ EDA completado exitosamente!")